# Section II v2 — improving the ticket router
**Runtime → Change runtime type → T4 GPU.** Then upload `finetune_v2_colab.zip` (from your project folder) using the Files panel on the left.

What changed vs v1 (DistilBERT, 41.6% accuracy):
1. **All the data**: v1 capped each class at 1,500 tickets; v2 uses every English ticket in the top 8 queues.
2. **Cleaner labels**: duplicate tickets and tickets that appear with *conflicting* queues are removed.
3. **Stronger backbone**: `roberta-base` (125M params, still a small language model) instead of DistilBERT (66M).
4. **Longer training**: up to 5 epochs with early stopping (v1 was still improving at epoch 3).
5. **Class-weighted loss** for the imbalanced full dataset.

Two experiments: **A** keeps all 8 queues. **B** merges the near-identical queues (IT Support + Technical Support, Customer Service + Product Support) into 6 routing groups.

Total time is about 50–60 minutes. If you're short on time, run only experiment B.

In [ ]:
!unzip -o -q finetune_v2_colab.zip -d /content/sdc
%cd /content/sdc
!pip -q install -r finetune/requirements.txt
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch runtime to T4!')

## Experiment A — 8 queues, roberta-base, full data

In [ ]:
!python finetune/prepare_data.py --run v2_8cls

In [ ]:
!python finetune/train_classifier.py --run v2_8cls --base roberta-base --epochs 5 --lr 2e-5 --batch 32

In [ ]:
!python finetune/evaluate.py --run v2_8cls --zero-shot-sample 400

## Experiment B — 6 merged routing groups, roberta-base, full data

In [ ]:
!python finetune/prepare_data.py --run v2_6cls_merged --merge

In [ ]:
!python finetune/train_classifier.py --run v2_6cls_merged --base roberta-base --epochs 5 --lr 2e-5 --batch 32

In [ ]:
!python finetune/evaluate.py --run v2_6cls_merged --zero-shot-sample 400

## Compare runs and plot (screenshot these for the report)

In [ ]:
!python finetune/compare_runs.py
from IPython.display import Image, display
for r in ['v2_8cls', 'v2_6cls_merged']:
    try: display(Image(f'finetune/results/{r}/confusion_matrix.png', width=650))
    except Exception as e: print(r, e)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, r in zip(axes, ['v2_8cls', 'v2_6cls_merged']):
    try:
        log = pd.read_csv(f'finetune/outputs/{r}/training_log.csv')
        log.dropna(subset=['loss']).plot(x='epoch', y='loss', ax=ax, label='train loss')
        log.dropna(subset=['eval_loss']).plot(x='epoch', y='eval_loss', ax=ax, label='val loss', marker='o')
        ax.set_title(r)
    except FileNotFoundError: pass
plt.tight_layout(); plt.savefig('finetune/results/v2_loss_curves.png', dpi=150); plt.show()

## Download results (+ the best model) back to your project folder
Set `BEST` to the run you want the app to use. Unzip into `~/Projects/servicedesk-copilot` on your Mac.

In [ ]:
BEST = 'v2_6cls_merged'   # or 'v2_8cls'
!rm -f v2_results.zip && zip -qr v2_results.zip finetune/results data/processed/*/labels.json finetune/outputs/$BEST -x '*/checkpoint-*'
from google.colab import files; files.download('v2_results.zip')

## (Optional) Publish the best model to the Hugging Face Hub for the live Space

In [ ]:
from huggingface_hub import notebook_login; notebook_login()
from transformers import AutoModelForSequenceClassification, AutoTokenizer
repo = 'vaikunths/roberta-ticket-router'
path = f'finetune/outputs/{BEST}'
AutoModelForSequenceClassification.from_pretrained(path).push_to_hub(repo)
AutoTokenizer.from_pretrained(path).push_to_hub(repo)